# Búsqueda vectorial SQL en IRIS

En el Lab 01 los vectores vivían en memoria. Aquí guardaremos los mismos chunks y embeddings en IRIS, crearemos un índice HNSW y haremos la búsqueda mediante SQL. Ejecuta las celdas en orden: hay dos que escriben datos en la base de datos.

## 1. Preparar las herramientas

Usamos `learning_ai_rag`, la biblioteca Python pequeña del curso. Sus piezas siguen siendo visibles: leer documentos, crear chunks, generar embeddings y usar un adaptador SQL para IRIS.

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

root = Path.cwd().resolve()
while not (root / 'packages').exists():
    root = root.parent

load_dotenv(root / '.env')
sys.path.insert(0, str(root / 'packages' / 'learning_ai_rag'))

from learning_ai_rag import (  # noqa: E402
    OpenRouterAnswerGenerator,
    RAGPipeline,
    SentenceTransformerEmbedder,
    load_markdown_documents,
)
from learning_ai_rag.ingest import chunk_documents  # noqa: E402
from learning_ai_rag.iris_vector import (  # noqa: E402
    INDEX_NAME,
    TABLE_NAME,
    VECTOR_DIMENSION,
    IrisConnectionSettings,
    IrisVectorStore,
)


## 2. Volver a las fuentes

La base de datos no inventa documentos nuevos: cargaremos las mismas políticas Markdown del Lab 01.

In [ ]:
documents = load_markdown_documents(root / 'datos-demo' / 'knowledge')
for document in documents:
    print(f'- {document.source}: {document.title}')

chunks = chunk_documents(documents, chunk_size_words=120, overlap_words=30)
print(f'\nChunks que se guardarán en IRIS: {len(chunks)}')
print(f'[{chunks[0].chunk_id}] {chunks[0].text[:280]}...')


## 3. Crear los embeddings locales

El modelo sigue ejecutándose en Python. IRIS recibe los vectores ya calculados y los almacena en una columna `VECTOR(DOUBLE, 384)`. No vamos a imprimir un vector entero: ver su tamaño ayuda. Leer 384 números no tanto.

In [ ]:
embedder = SentenceTransformerEmbedder()
if embedder.dimension != VECTOR_DIMENSION:
    raise ValueError(f'Este lab necesita {VECTOR_DIMENSION} dimensiones, no {embedder.dimension}.')

chunk_embeddings = embedder.encode([chunk.text for chunk in chunks])
print(f'Modelo: {embedder.model_name}')
print(f'Matriz de embeddings: {chunk_embeddings.shape}')


## 4. Conectar con IRIS

Comprueba que las variables `IRIS_HOST`, `IRIS_PORT`, `IRIS_NAMESPACE` e `IRIS_USERNAME` apuntan a tu instancia. No mostramos la contraseña.

In [ ]:
settings = IrisConnectionSettings.from_environment()
print(f'IRIS: {settings.host}:{settings.port} / {settings.namespace} como {settings.username}')
store = IrisVectorStore.connect(settings)
print('Conexión creada.')


## 5. Inicializar la estructura en IRIS ⚠️

Esta celda **escribe** en IRIS: crea la tabla `Learning_RAG.ChunkEmbedding` con una columna `VECTOR(DOUBLE, 384)` y el índice `ChunkEmbeddingHnsw` con distancia coseno. Ejecútala una sola vez en una instancia nueva. Si ya hiciste este lab, salta a la siguiente celda.

In [ ]:
store.initialize()
print(f'Tabla creada: {TABLE_NAME}')
print(f'Índice HNSW creado: {INDEX_NAME}')


## 6. Cargar el corpus ⚠️

Esta celda también **escribe** en IRIS. Inserta los chunks junto con sus embeddings mediante `TO_VECTOR(..., DOUBLE, 384)`, para que el tipo y la dimensión coincidan con la columna. Puedes repetirla si cambias el corpus.

In [ ]:
store.replace_chunks(chunks, chunk_embeddings)
print(f'{len(chunks)} chunks cargados o actualizados en IRIS.')


## 7. La consulta vectorial

La pregunta también se convierte en embedding. IRIS calcula `VECTOR_COSINE` dentro de SQL. `TOP` y `ORDER BY score DESC` permiten utilizar el índice HNSW. No recuperamos el vector crudo porque no aporta nada al resultado.

In [ ]:
print(f'''SELECT TOP 3 ChunkId, SourceName, ChunkText,
       VECTOR_COSINE(Embedding, TO_VECTOR(?, DOUBLE, {VECTOR_DIMENSION})) AS score
FROM {TABLE_NAME}
ORDER BY score DESC''')


In [ ]:
question = '¿Cuándo puedo devolver un producto?'
query_embedding = embedder.encode([question])[0]
results = store.search_by_embedding(query_embedding, top_k=3)

print(f'Pregunta: {question}\n')
for result in results:
    print(f'[{result.chunk.chunk_id}] {result.chunk.source} ({result.score:.3f})')
    print(f'{result.chunk.text[:260]}...\n')


## 8. Construir el RAG sobre IRIS

La búsqueda anterior ya devolvía los chunks desde IRIS. Ahora usamos el mismo `RAGPipeline` del Lab 01: transforma la pregunta en embedding, recupera evidencia desde IRIS, conserva las citas y se abstiene si no hay suficiente evidencia.

In [ ]:
pipeline = RAGPipeline(store, embedder, context_chunks=chunks)
evidence_answer = pipeline.answer(question)

print('Evidencia que recibirá el modelo:\n')
print(evidence_answer.answer)
print('\nCitas:')
for citation in evidence_answer.citations:
    print(f'- {citation.chunk.source} · {citation.chunk.chunk_id} ({citation.score:.3f})')


## 9. Generar una respuesta RAG (opcional)

Si has configurado `OPENROUTER_API_KEY`, esta celda entrega la evidencia recuperada desde IRIS al modelo. Sin clave, todo lo anterior sigue siendo el RAG de recuperación persistente. Simplemente no se redactará una respuesta.

In [ ]:
if not os.environ.get('OPENROUTER_API_KEY'):
    print('Define OPENROUTER_API_KEY para ejecutar esta celda.')
else:
    generator = OpenRouterAnswerGenerator()
    rag_answer = pipeline.answer_with_generator(question, generator)
    print('Respuesta RAG:\n')
    print(rag_answer.answer)
    print('\nCitas controladas por la aplicación:')
    for citation in rag_answer.citations:
        print(f'- {citation.chunk.source} · {citation.chunk.chunk_id}')


## Repetir desde terminal

Tras completar el recorrido, puedes hacer lo mismo de forma compacta:

```bash
python labs/02-sql-vector-search/vector_search.py --initialize --load
python labs/02-sql-vector-search/vector_search.py --question \"¿Cuándo puedo devolver un producto?\"
python labs/02-sql-vector-search/vector_search.py --question \"¿Cuándo puedo devolver un producto?\" --generate
```

En una tabla ya inicializada, usa sólo `--load` para actualizar el corpus.